# Test 7: Imbalanced Class Evaluation (Real-World Deployment Simulation)

**Goal**: Simulate a realistic deployment scenario where the class ratio is **90% safe / 10% malicious** (instead of the 50/50 training balance).

**Why this matters**: A model trained on balanced data can have inflated accuracy when evaluated on balanced data. In real APK scanning, the vast majority of apps are safe. Reviewers will ask *"does this hold up in practice?"*.

**Method**: No retraining needed. We resample the existing test set to approximate a 90/10 ratio by keeping all malicious samples and randomly downsampling safe samples. We then report precision, recall, F1, and FPR at the optimal threshold.

**Models evaluated**:
- GraphCodeBERT + DFG  (best single model)
- Ensemble (50/50 soft vote)

**Kaggle inputs**:
- `/kaggle/input/dfgdataset2/dataset_graphcodebert.jsonl`
- `/kaggle/input/model2/model.bin` (GraphCodeBERT weights)
- `/kaggle/input/model-codebert/model_codebert.bin`

**Outputs**: `test7_imbalanced_results.txt`, `test7_precision_recall_bar.png`

**Estimated runtime**: ~15–20 min on GPU (inference only, no training)

In [1]:
!pip install torch transformers tree_sitter==0.21.3 scikit-learn matplotlib -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 8.7 MB/s eta 0:00:00


In [2]:
import os, json, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader, Dataset, random_split
from transformers import RobertaConfig, RobertaModel, AutoTokenizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm
print('Imports OK')
print(f'CUDA: {torch.cuda.is_available()}')

Imports OK
CUDA: True


In [3]:
# ─── CONFIGURATION ────────────────────────────────────────────
class Args:
    train_file       = '/kaggle/input/datasets/hasanmahmudabdullah/dfgdataset2/dataset_graphcodebert.jsonl'
    gcb_weights      = '/kaggle/input/datasets/hasanmahmudabdullah3/gcbmodel/saved_models/model.bin'
    codebert_weights = '/kaggle/input/datasets/hasanmahmudabdullah3/codebertv2/saved_models_codebert/model.bin'
    code_length      = 384
    data_flow_length = 128
    eval_batch_size  = 32
    seed             = 42
    # Imbalance ratio to simulate
    target_malicious_ratio = 0.10   # 10% malicious, 90% safe
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    n_gpu  = torch.cuda.device_count()
args = Args()

OPT_THRESHOLD = 0.5

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if args.n_gpu > 0: torch.cuda.manual_seed_all(s)
set_seed(args.seed)
print(f'Device: {args.device}')
print(f'Threshold: {OPT_THRESHOLD}')
print(f'Target ratio: {args.target_malicious_ratio*100:.0f}% malicious')

Device: cuda
Threshold: 0.5
Target ratio: 10% malicious


In [4]:
# ─── MODEL DEFINITIONS (same as Test 2) ───────────────────────
class ModelA(nn.Module):
    def __init__(self, encoder, config, tokenizer, args):
        super().__init__()
        self.encoder = encoder; self.config = config
        self.tokenizer = tokenizer; self.args = args
        self.dropout    = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, 2)
    def forward(self, input_ids=None, p_ids=None, attn_mask=None, labels=None):
        ext_mask = (1.0 - attn_mask) * -10000.0
        ext_mask = ext_mask.unsqueeze(1)
        emb = self.encoder.embeddings(input_ids=input_ids, position_ids=p_ids)
        out = self.encoder.encoder(
            emb, attention_mask=ext_mask,
            head_mask=[None]*self.config.num_hidden_layers)[0]
        logits = self.classifier(self.dropout(out[:, 0, :]))
        prob   = F.softmax(logits, dim=-1)
        if labels is not None:
            return CrossEntropyLoss()(logits, labels), prob
        return prob

class ModelB(nn.Module):
    def __init__(self, encoder, config, tokenizer, args):
        super().__init__()
        self.encoder = encoder; self.config = config
        self.dropout    = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, 2)
    def forward(self, input_ids=None, attention_mask=None, labels=None):
        out    = self.encoder(input_ids=input_ids, attention_mask=attention_mask)[0]
        logits = self.classifier(self.dropout(out[:, 0, :]))
        prob   = F.softmax(logits, dim=-1)
        if labels is not None:
            return CrossEntropyLoss()(logits, labels), prob
        return prob

In [5]:
# ─── DATASET CLASSES (same as Test 2) ─────────────────────────
class TextDataset(Dataset):
    def __init__(self, tokenizer, args, file_path):
        self.args = args; self.tokenizer = tokenizer
        self.total_len = args.code_length + args.data_flow_length
        with open(file_path) as f:
            self.lines = f.readlines()
    def __len__(self): return len(self.lines)
    def _char_idx(self, lines, coord):
        r, c = coord
        return sum(len(lines[i]) for i in range(min(r,len(lines)))) + c
    def __getitem__(self, idx):
        entry = json.loads(self.lines[idx])
        code  = entry.get('code', '')
        dfg   = entry.get('dfg', [])[:self.args.data_flow_length]
        label = int(entry.get('label', 0))
        code_lines = code.splitlines(keepends=True)
        tok = self.tokenizer(
            code, max_length=self.args.code_length, truncation=True,
            padding='max_length', return_offsets_mapping=True)
        input_ids, offsets = tok['input_ids'], tok['offset_mapping']
        dfg_ids = [self.tokenizer.unk_token_id]*len(dfg)
        n2t, p2n = {}, {}
        for ni, node in enumerate(dfg):
            sp, ep = node[1][0], node[1][1]
            pk = (sp[0],sp[1],ep[0],ep[1]); p2n[pk]=ni
            try:
                cs=self._char_idx(code_lines,sp); ce=self._char_idx(code_lines,ep)
                n2t[ni]=[i for i,(ts,te) in enumerate(offsets)
                         if ts!=te and ((ts>=cs and te<=ce) or (cs>=ts and ce<=te))]
            except IndexError:
                n2t[ni]=[]
        c_len = self.args.code_length
        mask  = np.zeros((self.total_len, self.total_len), dtype=bool)
        mask[:c_len,:c_len] = True
        for ni, node in enumerate(dfg):
            ani=c_len+ni
            for ti in n2t.get(ni,[]): mask[ani,ti]=mask[ti,ani]=True
            for pp in node[4]:
                pk=(pp[0][0],pp[0][1],pp[1][0],pp[1][1])
                if pk in p2n: ap=c_len+p2n[pk]; mask[ani,ap]=mask[ap,ani]=True
            mask[ani,ani]=True
        ids = input_ids+dfg_ids
        p_ids=[i+2 for i in range(c_len)]+[0]*len(dfg_ids)
        pad=self.total_len-len(ids)
        if pad>0: ids+=[self.tokenizer.pad_token_id]*pad; p_ids+=[1]*pad
        return {
            'input_ids': torch.tensor(ids,   dtype=torch.long),
            'p_ids':     torch.tensor(p_ids, dtype=torch.long),
            'attn_mask': torch.tensor(mask,  dtype=torch.float),
            'label':     torch.tensor(label, dtype=torch.long)
        }

class SimpleCodeDataset(Dataset):
    def __init__(self, tokenizer, args, file_path):
        self.tokenizer=tokenizer; self.args=args
        with open(file_path) as f: self.lines=f.readlines()
    def __len__(self): return len(self.lines)
    def __getitem__(self, idx):
        try: entry=json.loads(self.lines[idx])
        except: return self.__getitem__((idx+1)%len(self.lines))
        code=entry.get('code',''); label=int(entry.get('label',0))
        tok=self.tokenizer(code,max_length=self.args.code_length,
                           truncation=True,padding='max_length')
        return {
            'input_ids':      torch.tensor(tok['input_ids'],      dtype=torch.long),
            'attention_mask': torch.tensor(tok['attention_mask'], dtype=torch.long),
            'label':          torch.tensor(label,                 dtype=torch.long)
        }

In [6]:
# ─── LOAD MODELS ──────────────────────────────────────────────
print('Loading Model A (GraphCodeBERT)...')
cfg_a = RobertaConfig.from_pretrained('microsoft/graphcodebert-base'); cfg_a.num_labels=2
tok_a = AutoTokenizer.from_pretrained('microsoft/graphcodebert-base', use_fast=True)
enc_a = RobertaModel.from_pretrained('microsoft/graphcodebert-base', config=cfg_a)
model_a = ModelA(enc_a, cfg_a, tok_a, args).to(args.device)
model_a.load_state_dict(torch.load(args.gcb_weights, map_location=args.device))
model_a.eval(); print('  ✓ Model A loaded')

has_b = os.path.exists(args.codebert_weights)
if has_b:
    print('Loading Model B (CodeBERT)...')
    cfg_b = RobertaConfig.from_pretrained('microsoft/codebert-base'); cfg_b.num_labels=2
    tok_b = AutoTokenizer.from_pretrained('microsoft/codebert-base', use_fast=True)
    enc_b = RobertaModel.from_pretrained('microsoft/codebert-base', config=cfg_b)
    model_b = ModelB(enc_b, cfg_b, tok_b, args).to(args.device)
    model_b.load_state_dict(torch.load(args.codebert_weights, map_location=args.device))
    model_b.eval(); print('  ✓ Model B loaded')
else:
    print(f'  ⚠ CodeBERT not found at {args.codebert_weights} — ensemble will be skipped')
    model_b = None

Loading Model A (GraphCodeBERT)...


config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

  ✓ Model B loaded


In [7]:
# ─── BUILD TEST SET ───────────────────────────────────────────
# Use the same 90/10 train/test split — analyse the held-out test set
print('Building test dataset...')
ds_a = TextDataset(tok_a, args, args.train_file)
n    = len(ds_a)
train_n = int(0.9*n); test_n = n-train_n
gen = torch.Generator().manual_seed(args.seed)
_, test_ds_a = random_split(ds_a, [train_n, test_n], generator=gen)
if model_b:
    ds_b = SimpleCodeDataset(tok_b, args, args.train_file)
    gen  = torch.Generator().manual_seed(args.seed)
    _, test_ds_b = random_split(ds_b, [train_n, test_n], generator=gen)
print(f'Test samples: {test_n:,}')


Building test dataset...
Test samples: 19,996


In [8]:
# ─── RUN INFERENCE ON BALANCED SET ────────────────────────────
@torch.no_grad()
def get_probs_a(model, dataset):
    loader = DataLoader(dataset, batch_size=args.eval_batch_size, shuffle=False, num_workers=2)
    all_p, all_l = [], []
    for batch in tqdm(loader, desc='Inference A'):
        inp = {k: batch[k].to(args.device) for k in ('input_ids','p_ids','attn_mask')}
        pr  = model(**inp)[:, 1].cpu().numpy()
        all_p.extend(pr); all_l.extend(batch['label'].numpy())
    return np.array(all_p), np.array(all_l)

@torch.no_grad()
def get_probs_b(model, dataset):
    loader = DataLoader(dataset, batch_size=args.eval_batch_size, shuffle=False, num_workers=2)
    all_p, all_l = [], []
    for batch in tqdm(loader, desc='Inference B'):
        inp = {k: batch[k].to(args.device) for k in ('input_ids','attention_mask')}
        pr  = model(**inp)[:, 1].cpu().numpy()
        all_p.extend(pr); all_l.extend(batch['label'].numpy())
    return np.array(all_p), np.array(all_l)

probs_a, labels_bal = get_probs_a(model_a, test_ds_a)
if model_b:
    probs_b, _ = get_probs_b(model_b, test_ds_b)
    probs_ens  = (probs_a + probs_b) / 2.0
print('Inference complete')
print(f'Test set  -> {len(labels_bal):,} samples, {labels_bal.mean():.4f} malicious ratio')

Inference B: 100%|██████████| 625/625 [03:47<00:00,  2.74it/s]

Inference complete
Test set  -> 19,996 samples, 0.4951 malicious ratio


In [9]:
rng = np.random.default_rng(42)

safe_idx = np.where(labels_bal == 0)[0]
mal_idx  = np.where(labels_bal == 1)[0]

# Keep all safe, downsample malicious to ~10% of total
n_safe       = len(safe_idx)
n_mal_target = int(n_safe * args.target_malicious_ratio /
                   (1 - args.target_malicious_ratio))
n_mal_target = min(n_mal_target, len(mal_idx))
mal_sample   = rng.choice(mal_idx, size=n_mal_target, replace=False)
imb_idx      = np.concatenate([safe_idx, mal_sample])

probs_a_imb  = probs_a[imb_idx]
labels_imb   = labels_bal[imb_idx]
if model_b is not None:
    probs_ens_imb = probs_ens[imb_idx]

print(f'\nImbalanced set -> {len(labels_imb):,} samples')
print(f'  Malicious: {labels_imb.sum():,}  ({labels_imb.mean()*100:.1f}%)')
print(f'  Safe:      {(labels_imb==0).sum():,}  ({(1-labels_imb.mean())*100:.1f}%)')


Imbalanced set -> 11,216 samples
  Malicious: 1,121  (10.0%)
  Safe:      10,095  (90.0%)


In [10]:
# ─── EVALUATION HELPER ────────────────────────────────────────
def evaluate(probs, labels, name, threshold=OPT_THRESHOLD):
    preds = (probs >= threshold).astype(int)
    acc   = accuracy_score(labels, preds)
    prec  = precision_score(labels, preds, zero_division=0)
    rec   = recall_score(labels, preds, zero_division=0)
    f1    = f1_score(labels, preds, zero_division=0)
    auc   = roc_auc_score(labels, probs)
    pr_a  = average_precision_score(labels, probs)
    cm    = confusion_matrix(labels, preds)
    fn    = cm[1, 0] if cm.shape == (2,2) else 0
    fp    = cm[0, 1] if cm.shape == (2,2) else 0
    fpr   = fp / max((cm[0,0]+cm[0,1]), 1)
    print(f'\n=== {name} (threshold={threshold}) ===')
    print(f'  Accuracy  : {acc:.4%}')
    print(f'  Precision : {prec:.4f}  (of predicted malicious, how many actually are)')
    print(f'  Recall    : {rec:.4f}  (of actual malicious, how many we caught)')
    print(f'  F1        : {f1:.4f}')
    print(f'  ROC-AUC   : {auc:.4f}')
    print(f'  PR-AUC    : {pr_a:.4f}')
    print(f'  FN        : {fn:,}  (missed malware)')
    print(f'  FP        : {fp:,}  (false alarms)')
    print(f'  FPR       : {fpr:.4f}  (false alarm rate on safe apps)')
    return dict(name=name, acc=acc, prec=prec, rec=rec, f1=f1,
                auc=auc, pr_auc=pr_a, fn=fn, fp=fp, fpr=fpr)

In [11]:
# ─── EVALUATE: BALANCED (original 50/50) ──────────────────────
print('\n--- BALANCED (50/50) EVALUATION ---')
res_bal_a = evaluate(probs_a, labels_bal, 'GraphCodeBERT+DFG [balanced 50/50]')
if model_b:
    res_bal_ens = evaluate(probs_ens, labels_bal, 'Ensemble [balanced 50/50]')


--- BALANCED (50/50) EVALUATION ---

=== GraphCodeBERT+DFG [balanced 50/50] (threshold=0.5) ===
  Accuracy  : 88.7077%
  Precision : 0.8903  (of predicted malicious, how many actually are)
  Recall    : 0.8804  (of actual malicious, how many we caught)
  F1        : 0.8853
  ROC-AUC   : 0.9616
  PR-AUC    : 0.9622
  FN        : 1,184  (missed malware)
  FP        : 1,074  (false alarms)
  FPR       : 0.1064  (false alarm rate on safe apps)

=== Ensemble [balanced 50/50] (threshold=0.5) ===
  Accuracy  : 88.8828%
  Precision : 0.8884  (of predicted malicious, how many actually are)
  Recall    : 0.8869  (of actual malicious, how many we caught)
  F1        : 0.8876
  ROC-AUC   : 0.9638
  PR-AUC    : 0.9644
  FN        : 1,120  (missed malware)
  FP        : 1,103  (false alarms)
  FPR       : 0.1093  (false alarm rate on safe apps)


In [12]:
# ─── EVALUATE: IMBALANCED (90/10) ─────────────────────────────
print('\n--- IMBALANCED (90% safe / 10% malicious) EVALUATION ---')
res_imb_a = evaluate(probs_a_imb, labels_imb, 'GraphCodeBERT+DFG [imbalanced 90/10]')
if model_b:
    res_imb_ens = evaluate(probs_ens_imb, labels_imb, 'Ensemble [imbalanced 90/10]')


--- IMBALANCED (90% safe / 10% malicious) EVALUATION ---

=== GraphCodeBERT+DFG [imbalanced 90/10] (threshold=0.5) ===
  Accuracy  : 89.1494%
  Precision : 0.4766  (of predicted malicious, how many actually are)
  Recall    : 0.8724  (of actual malicious, how many we caught)
  F1        : 0.6165
  ROC-AUC   : 0.9596
  PR-AUC    : 0.8131
  FN        : 143  (missed malware)
  FP        : 1,074  (false alarms)
  FPR       : 0.1064  (false alarm rate on safe apps)

=== Ensemble [imbalanced 90/10] (threshold=0.5) ===
  Accuracy  : 88.9622%
  Precision : 0.4720  (of predicted malicious, how many actually are)
  Recall    : 0.8796  (of actual malicious, how many we caught)
  F1        : 0.6143
  ROC-AUC   : 0.9609
  PR-AUC    : 0.8217
  FN        : 135  (missed malware)
  FP        : 1,103  (false alarms)
  FPR       : 0.1093  (false alarm rate on safe apps)


In [13]:
# ─── THRESHOLD SENSITIVITY ON IMBALANCED SET ──────────────────
# Shows how precision/recall trade off changes under real-world imbalance
print('\nThreshold sensitivity (GraphCodeBERT+DFG, imbalanced 90/10 set):')
print(f'{"Threshold":>10} {"Precision":>10} {"Recall":>8} {"F1":>8} {"FPR":>8} {"FN":>6}')
print('-'*55)
for th in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65]:
    p = (probs_a_imb >= th).astype(int)
    prec = precision_score(labels_imb, p, zero_division=0)
    rec  = recall_score(labels_imb,    p, zero_division=0)
    f1   = f1_score(labels_imb,        p, zero_division=0)
    cm   = confusion_matrix(labels_imb, p)
    fn   = cm[1,0] if cm.shape==(2,2) else 0
    fp   = cm[0,1] if cm.shape==(2,2) else 0
    fpr  = fp / max(cm[0,0]+cm[0,1], 1)
    print(f'{th:>10.2f} {prec:>10.4f} {rec:>8.4f} {f1:>8.4f} {fpr:>8.4f} {fn:>6,}')


Threshold sensitivity (GraphCodeBERT+DFG, imbalanced 90/10 set):
 Threshold  Precision   Recall       F1      FPR     FN
-------------------------------------------------------
      0.30     0.3936   0.9153   0.5504   0.1566     95
      0.35     0.4079   0.9072   0.5628   0.1462    104
      0.40     0.4253   0.8965   0.5769   0.1345    116
      0.45     0.4471   0.8858   0.5943   0.1216    128
      0.50     0.4766   0.8724   0.6165   0.1064    143
      0.55     0.5104   0.8555   0.6393   0.0911    162
      0.60     0.5439   0.8341   0.6585   0.0777    186
      0.65     0.5765   0.8171   0.6760   0.0667    205


In [14]:
# ─── COMPARISON BAR CHART ─────────────────────────────────────
conditions = ['Balanced\n(50/50)', 'Imbalanced\n(90/10 real-world)']
has_ens = model_b is not None

if has_ens:
    prec_vals = [res_bal_a['prec'],  res_imb_a['prec'],  res_bal_ens['prec'],  res_imb_ens['prec']]
    rec_vals  = [res_bal_a['rec'],   res_imb_a['rec'],   res_bal_ens['rec'],   res_imb_ens['rec']]
    f1_vals   = [res_bal_a['f1'],    res_imb_a['f1'],    res_bal_ens['f1'],    res_imb_ens['f1']]
    xlabels   = ['GCB+DFG\nBalanced','GCB+DFG\nImbalanced','Ensemble\nBalanced','Ensemble\nImbalanced']
else:
    prec_vals = [res_bal_a['prec'],  res_imb_a['prec']]
    rec_vals  = [res_bal_a['rec'],   res_imb_a['rec']]
    f1_vals   = [res_bal_a['f1'],    res_imb_a['f1']]
    xlabels   = ['GCB+DFG\nBalanced','GCB+DFG\nImbalanced']

x = np.arange(len(xlabels))
w = 0.26
fig, ax = plt.subplots(figsize=(11, 6))
b1 = ax.bar(x - w,   prec_vals, w, label='Precision', color='#4C72B0', alpha=0.88)
b2 = ax.bar(x,       rec_vals,  w, label='Recall',    color='#55A868', alpha=0.88)
b3 = ax.bar(x + w,   f1_vals,   w, label='F1',        color='#DD8452', alpha=0.88)

for bars in [b1, b2, b3]:
    for bar in bars:
        ax.annotate(f'{bar.get_height():.3f}',
                    xy=(bar.get_x()+bar.get_width()/2, bar.get_height()),
                    xytext=(0,4), textcoords='offset points',
                    ha='center', va='bottom', fontsize=9)
ax.set_ylim(0.0, 1.12)
ax.set_xticks(x)
ax.set_xticklabels(xlabels, fontsize=10)
ax.set_ylabel('Score', fontsize=13)
ax.set_title(f'Precision / Recall / F1: Balanced vs Imbalanced (threshold={OPT_THRESHOLD})', fontsize=13)
ax.legend(fontsize=11)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)
out_img = '/kaggle/working/test7_precision_recall_bar.png'
fig.savefig(out_img, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved -> {out_img}')

Saved -> /kaggle/working/test7_precision_recall_bar.png


In [15]:
# ─── SAVE TEXT RESULTS ────────────────────────────────────────
out_txt = '/kaggle/working/test7_imbalanced_results.txt'
with open(out_txt, 'w') as fh:
    fh.write('Test 7: Imbalanced Class Evaluation\n')
    fh.write('='*60 + '\n')
    fh.write(f'Threshold used: {OPT_THRESHOLD}\n')
    fh.write(f'Imbalanced ratio: 90% safe / 10% malicious\n\n')
    for res in ([res_bal_a, res_imb_a] +
                ([res_bal_ens, res_imb_ens] if has_ens else [])):
        fh.write(f"\n{res['name']}\n")
        for k in ['acc','prec','rec','f1','auc','pr_auc','fn','fp','fpr']:
            fh.write(f'  {k:<8}: {res[k]:.4f}\n' if isinstance(res[k],float)
                     else f'  {k:<8}: {res[k]:,}\n')
print(f'Saved -> {out_txt}')
print(f'Saved -> {out_img}')

Saved -> /kaggle/working/test7_imbalanced_results.txt
Saved -> /kaggle/working/test7_precision_recall_bar.png
